# Notebook 01 — Tracking Pipeline

This notebook walks through every stage of the tracking pipeline interactively.
It is equivalent to running `scripts/run_tracking.py` but lets you inspect
intermediate outputs at each step.

## Stages
1. Setup & configuration
2. (Optional) Background subtraction
3. Draw vial ROIs
4. RF-DETR + OC-SORT tracking → wide CSV
5. Hungarian stitching → stitched long CSV
6. Vial assignment + compact IDs → compact_tracks.csv
7. Overlay video rendering

**Replace all `PLACEHOLDER` paths with your actual file paths.**

In [ ]:
import sys
sys.path.insert(0, '..')  # so 'src' is importable from the notebooks/ folder

import os
import cv2
import pandas as pd
from pathlib import Path
from IPython.display import Video

from src.preprocessing import preprocess_bgsub_gui_cv2_avg_background
from src.tracking import export_tracks_xy_tuple_csv_one_config
from src.stitching import stitch_wide_csv_to_long, wide_to_long
from src.roi import draw_and_save_vial_rois, assign_vials_and_compact_ids
from src.visualization import render_vial_overlay_video

## 1 — Configuration

Set your paths and Roboflow credentials here.

In [ ]:
# ---- EDIT THESE ----
RAW_VIDEO    = "PLACEHOLDER/my_experiment.mp4"
OUTPUT_PATH  = "PLACEHOLDER/outputs/my_run"
API_KEY      = "PLACEHOLDER_ROBOFLOW_API_KEY"
MODEL_ID     = "PLACEHOLDER_MODEL_ID"   # e.g. "flies-123/1"

# Tracker params (defaults match config.yaml)
confidence              = 0.10
lost_track_buffer       = 90
min_matching_threshold  = 0.01
min_consecutive_frames  = 10

os.makedirs(OUTPUT_PATH, exist_ok=True)
PATH_TO_VID = RAW_VIDEO
print("Output dir:", OUTPUT_PATH)

## 2 — (Optional) Background subtraction

Opens an OpenCV GUI: draw a crop ROI and choose a frame range.
The output is a `_pp.mp4` file with the average background subtracted.
Skip this cell if your video already has good contrast.

In [ ]:
preprocess = False  # set to True to run the GUI

if preprocess:
    PATH_TO_VID = Path(
        preprocess_bgsub_gui_cv2_avg_background(
            video_path=str(RAW_VIDEO),
            out_mp4=None,
            default_end=700,
            bg_sample_stride=1,
        )
    )
    print("Preprocessed video:", PATH_TO_VID)

## 3 — Draw vial ROIs

Opens an OpenCV GUI on frame 0: drag rectangles around each vial.
Press **q** when all 6 ROIs are drawn. Saved to `vial_rois.json`.

This is a one-time step — reuse the JSON for the same experimental setup.

In [ ]:
ROI_JSON = os.path.join(OUTPUT_PATH, "vial_rois.json")

draw_and_save_vial_rois(
    video_path=str(PATH_TO_VID),
    roi_json_path=ROI_JSON,
)

## 4 — RF-DETR + OC-SORT tracking

Runs the detector + tracker on every frame and writes a wide CSV.
This is the most time-consuming step. 

In [ ]:
WIDE_CSV = os.path.join(OUTPUT_PATH, "tracks_wide_format.csv")

df_wide = export_tracks_xy_tuple_csv_one_config(
    video_path=str(PATH_TO_VID),
    output_csv=WIDE_CSV,
    api_key=API_KEY,
    model_id=MODEL_ID,
    confidence=confidence,
    lost_track_buffer=lost_track_buffer,
    minimum_matching_threshold=min_matching_threshold,
    minimum_consecutive_frames=min_consecutive_frames,
    max_frames=None,
)

print(df_wide.shape)
df_wide.head()

## 5 — Hungarian stitching

Links fragmented tracklets across gaps using motion-consistent assignment.
Output: long CSV with `orig_id` and `stitched_id` columns.

In [ ]:
STITCHED_CSV = os.path.join(OUTPUT_PATH, "tracks_xy_stitched_long.csv")

stats = stitch_wide_csv_to_long(
    input_csv=WIDE_CSV,
    output_stitched_long=STITCHED_CSV,
    max_gap=lost_track_buffer,
    gap_penalty=0.05,
)

print(stats)

## 6 — Vial assignment + compact IDs

Assigns each point to a vial using the ROI JSON, then assigns compact sequential IDs
(left → right within each vial).

In [ ]:
COMPACT_CSV = os.path.join(OUTPUT_PATH, "compact_tracks.csv")

cap = cv2.VideoCapture(str(PATH_TO_VID))
fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
cap.release()

df_compact = assign_vials_and_compact_ids(
    stitched_csv=STITCHED_CSV,
    roi_json=ROI_JSON,
    out_csv=COMPACT_CSV,
    fps=fps,
)

print(df_compact.shape)
df_compact.head()

## 7 — Overlay video

Renders each fly as a coloured dot on the original video.

In [ ]:
OVERLAY_MP4 = os.path.join(OUTPUT_PATH, "overlay_vials_shaded.mp4")

render_vial_overlay_video(
    video_path=str(PATH_TO_VID),
    csv_path=COMPACT_CSV,
    out_mp4=OVERLAY_MP4,
)

Video(OVERLAY_MP4, width=800)